# 객체 YOLO 모델 학습_Colab

로컬에서 만든 `Train2YOLO_Outdoor_ObjectBox.zip`을 Google Drive에 올린 뒤 Colab에서 `yolo26n` detection 모델을 학습합니다.

권장 Drive 구조:

```text
MyDrive/비전응용프로젝트/실외 보행 모델/객체/
├─ Train2YOLO_Outdoor_ObjectBox.zip
└─ runs/                         # 자동 생성
```


In [ ]:
# 필요 시 한 번만 실행
%pip install -U ultralytics pyyaml

In [ ]:
from pathlib import Path
import shutil
import yaml

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/비전응용프로젝트/실외 보행 모델/객체')
DRIVE_ZIP = DRIVE_ROOT / 'Train2YOLO_Outdoor_ObjectBox.zip'
LOCAL_ROOT = Path('/content/Train2YOLO_Outdoor_ObjectBox')
LOCAL_DATA_YAML = LOCAL_ROOT / 'data.yaml'
PROJECT_DIR = DRIVE_ROOT / 'runs'
RUN_NAME = 'outdoor_object_yolo26n'

IMGSZ = 512
EPOCHS = 50   # 총 학습 epoch 목표. 중단 후 이어 학습 시 이 epoch까지 진행합니다.
BATCH = 16
WORKERS = 2
DEVICE = 0
RESUME = True   # 중단된 학습을 이어가려면 True. 새로 시작하려면 False.

print('DRIVE_ZIP:', DRIVE_ZIP, DRIVE_ZIP.exists())
print('LOCAL_ROOT:', LOCAL_ROOT)
print('PROJECT_DIR:', PROJECT_DIR)
if not DRIVE_ZIP.exists():
    raise FileNotFoundError(DRIVE_ZIP)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


## 데이터셋 준비

In [ ]:
if not LOCAL_ROOT.exists():
    shutil.unpack_archive(str(DRIVE_ZIP), extract_dir='/content')

if not LOCAL_DATA_YAML.exists():
    candidates = list(Path('/content').glob('**/data.yaml'))
    print('data.yaml candidates:', candidates)
    if len(candidates) == 1:
        LOCAL_ROOT = candidates[0].parent
        LOCAL_DATA_YAML = candidates[0]
    else:
        raise FileNotFoundError('data.yaml을 찾지 못했습니다.')

with open(LOCAL_DATA_YAML, 'r', encoding='utf-8') as f:
    data = yaml.safe_load(f)
data['path'] = str(LOCAL_ROOT)
with open(LOCAL_DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

print(LOCAL_DATA_YAML.read_text(encoding='utf-8'))
for rel in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    folder = LOCAL_ROOT / rel
    pattern = '*' if rel.startswith('images') else '*.txt'
    print(rel, len(list(folder.glob(pattern))) if folder.exists() else 'MISSING')


## 학습 또는 이어서 학습

`RESUME=True`이고 `last.pt`가 있으면 이어서 학습합니다. 아직 한 번도 실행하지 않아 `last.pt`가 없으면 자동으로 새 학습을 시작합니다.


In [ ]:
from ultralytics import YOLO

run_dir = PROJECT_DIR / RUN_NAME
last_pt = run_dir / 'weights' / 'last.pt'
best_pt = run_dir / 'weights' / 'best.pt'
should_resume = RESUME and last_pt.exists()

print('RUN_NAME:', RUN_NAME)
print('run_dir:', run_dir)
print('RESUME:', RESUME)
print('last.pt:', last_pt, last_pt.exists())
print('best.pt:', best_pt, best_pt.exists())
print('should_resume:', should_resume)

if should_resume:
    model = YOLO(str(last_pt))
    # resume=True는 last.pt의 학습 상태에서 이어가며, epochs는 최종 목표 epoch입니다.
    results = model.train(resume=True, epochs=EPOCHS)
else:
    if RESUME:
        print('last.pt가 없어 새 학습을 시작합니다.')
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=str(LOCAL_DATA_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        workers=WORKERS,
        device=DEVICE,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=20,
        cache=False,
    )


## 결과 확인

In [ ]:
run_dir = PROJECT_DIR / RUN_NAME
for rel in ['weights/best.pt', 'weights/last.pt', 'results.csv', 'args.yaml']:
    p = run_dir / rel
    print(rel, p.exists(), p)

# 현재 몇 epoch까지 진행됐는지 간단히 확인합니다.
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    lines = [line.strip() for line in results_csv.read_text(encoding='utf-8').splitlines() if line.strip()]
    if len(lines) >= 2:
        print('\n마지막 기록:')
        print(lines[-1])
        try:
            last_epoch = int(float(lines[-1].split(',')[0].strip()))
            print('마지막 완료 epoch:', last_epoch)
        except Exception:
            pass
